# Task 4 海龟策略回测\n\n本 Notebook 使用复权后的 `qfq_*` 价格字段计算海龟策略。通道使用 `shift(1)`，信号采用“收盘确认、下一交易日开盘执行”，避免未来函数。\n

## 1. 策略思想\n\n海龟策略的核心思想是用高低点通道识别趋势突破，用 ATR/N 衡量波动，用 2N 止损和风险定仓控制单笔风险。它的优势在于规则透明、纪律性强、能够抓住持续趋势，同时也会在震荡市场中产生假突破。\n

In [ ]:
from pathlib import Path\nimport pandas as pd\nfrom turtle_strategy import TurtleStrategyConfig, add_turtle_indicators\nfrom backtester import TurtleBacktestConfig, run_turtle_backtest, calculate_performance_metrics\n\nPROJECT_ROOT = Path('..').resolve()\nDATA_PATH = PROJECT_ROOT / 'data/processed/equities/daily/smic_h_00981_HK_daily_20250703_20260703.csv'\nstrategy_config = TurtleStrategyConfig(entry_window=20, exit_window=10, n_window=20, stop_n=2.0)\nbacktest_config = TurtleBacktestConfig(initial_cash=100000, risk_per_unit=0.01, fee_rate=0.001, slippage_rate=0.0005)\n

In [ ]:
data = pd.read_csv(DATA_PATH, dtype={'trade_date': str, 'ts_code': str}, encoding='utf-8-sig')\ndata['date'] = pd.to_datetime(data['trade_date'], format='%Y%m%d')\ndata = data.sort_values('trade_date').reset_index(drop=True)\ndata.head()\n

## 2. 计算高低点通道与 ATR/N\n\n20 日高点通道用于入场，10 日低点通道用于退出。真实波幅 TR 取 `high-low`、`abs(high-pre_close)`、`abs(low-pre_close)` 的最大值，N 值使用海龟原版 20 日平滑。\n

In [ ]:
signals = add_turtle_indicators(data, strategy_config)\nsignals[['trade_date','qfq_close','tr','n_20','entry_high_20','exit_low_10','breakout_long','exit_long']].tail(10)\n

## 3. 交易信号与模拟回测\n\n当收盘价突破前一日已知的 20 日高点通道且当前无仓位时，下一交易日开盘买入；当收盘价跌破 10 日低点通道或触发 2N 止损时，下一交易日开盘卖出。\n

In [ ]:
portfolio = run_turtle_backtest(data, strategy_config, backtest_config)\nmetrics = calculate_performance_metrics(portfolio, backtest_config)\nmetrics\n

## 4. 可视化和参数敏感性\n\nNotebook 只保留策略计算、回测和结果读取代码；页面报告与课程文档由项目脚本单独生成，不放在 Notebook 代码单元中。\n

In [ ]:
summary = pd.read_csv('outputs/turtle_core_metrics_summary.csv')\nsummary[['label','cumulative_return','max_drawdown','sharpe_ratio','benchmark_cumulative_return','trade_count']]\n